In [12]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Notebook E-commerce")
    .master("local[*]")
    .getOrCreate()
)
#spark.sparkContext.setLogLevel("INFO")

In [ ]:
dados = [
    ("JP", "Arthur", 100),
    ("CG", "Bruna", 200),
    ("JP", "Carlos", 50),
    ("JP", "Daniel", 70),
    ("CG", "Camila", 150),
    ("PE", "Eduardo", 120),
    ("PE", "Fernanda", 130),
]

df = spark.createDataFrame(dados, ["regiao", "nome", "valor"])
df = df.repartition(4) # uma tarefa para 4 comandos

26/09/16 20:01:34 INFO SharedState: Setting hive.metastore.warehouse.dir ('null') to the value of spark.sql.warehouse.dir.
26/09/16 20:01:34 INFO SharedState: Warehouse path is 'file:/opt/spark/work-dir/spark-warehouse'.
26/09/16 20:01:34 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@2632d79e{/SQL,null,AVAILABLE,@Spark}
26/09/16 20:01:34 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@56be5c95{/SQL/json,null,AVAILABLE,@Spark}
26/09/16 20:01:34 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@437d88bb{/SQL/execution,null,AVAILABLE,@Spark}
26/09/16 20:01:34 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@1985c4b4{/SQL/execution/json,null,AVAILABLE,@Spark}
26/09/16 20:01:34 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@513162a2{/static/sql,null,AVAILABLE,@Spark}


In [4]:
df.show()

26/09/16 20:04:48 INFO CodeGenerator: Code generated in 166.045958 ms
26/09/16 20:04:48 INFO DAGScheduler: Registering RDD 7 (showString at <unknown>:0) as input to shuffle 0
26/09/16 20:04:48 INFO DAGScheduler: Got map stage job 0 (showString at <unknown>:0) with 16 output partitions
26/09/16 20:04:48 INFO DAGScheduler: Final stage: ShuffleMapStage 0 (showString at <unknown>:0)
26/09/16 20:04:48 INFO DAGScheduler: Parents of final stage: List()
26/09/16 20:04:48 INFO DAGScheduler: Missing parents: List()
26/09/16 20:04:48 INFO DAGScheduler: Submitting ShuffleMapStage 0 (MapPartitionsRDD[7] at showString at <unknown>:0), which has no missing parents
26/09/16 20:04:48 INFO MemoryStore: Block broadcast_0 stored as values in memory (estimated size 15.8 KiB, free 1048.8 MiB)
26/09/16 20:04:48 INFO MemoryStore: Block broadcast_0_piece0 stored as bytes in memory (estimated size 8.2 KiB, free 1048.8 MiB)
26/09/16 20:04:48 INFO BlockManagerInfo: Added broadcast_0_piece0 in memory on jupyter:37

+------+--------+-----+
|regiao|    nome|valor|
+------+--------+-----+
|    JP|  Arthur|  100|
|    JP|  Carlos|   50|
|    CG|   Bruna|  200|
|    JP|  Daniel|   70|
|    CG|  Camila|  150|
|    PE| Eduardo|  120|
|    PE|Fernanda|  130|
+------+--------+-----+



26/09/16 20:04:50 INFO CodeGenerator: Code generated in 23.399924 ms


In [7]:
df_filtrado = df.filter(F.col("valor") >= 80)

df_aggregado = (df
                .groupBy("regiao")
                .agg(F.sum("valor")
                .alias("total_valor"))
                .orderBy("total_valor")
                )

In [ ]:
df_filtrado.explain(True) #plano otimizado

== Parsed Logical Plan ==
'Filter ('valor >= 80)
+- Repartition 4, true
   +- LogicalRDD [regiao#0, nome#1, valor#2L], false

== Analyzed Logical Plan ==
regiao: string, nome: string, valor: bigint
Filter (valor#2L >= cast(80 as bigint))
+- Repartition 4, true
   +- LogicalRDD [regiao#0, nome#1, valor#2L], false

== Optimized Logical Plan ==
Repartition 4, true
+- Filter (isnotnull(valor#2L) AND (valor#2L >= 80))
   +- LogicalRDD [regiao#0, nome#1, valor#2L], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=41]
   +- Filter (isnotnull(valor#2L) AND (valor#2L >= 80))
      +- Scan ExistingRDD[regiao#0,nome#1,valor#2L]



In [9]:
df_aggregado.explain(True)

== Parsed Logical Plan ==
'Sort ['total_valor ASC NULLS FIRST], true
+- Aggregate [regiao#0], [regiao#0, sum(valor#2L) AS total_valor#23L]
   +- Repartition 4, true
      +- LogicalRDD [regiao#0, nome#1, valor#2L], false

== Analyzed Logical Plan ==
regiao: string, total_valor: bigint
Sort [total_valor#23L ASC NULLS FIRST], true
+- Aggregate [regiao#0], [regiao#0, sum(valor#2L) AS total_valor#23L]
   +- Repartition 4, true
      +- LogicalRDD [regiao#0, nome#1, valor#2L], false

== Optimized Logical Plan ==
Sort [total_valor#23L ASC NULLS FIRST], true
+- Aggregate [regiao#0], [regiao#0, sum(valor#2L) AS total_valor#23L]
   +- Repartition 4, true
      +- Project [regiao#0, valor#2L]
         +- LogicalRDD [regiao#0, nome#1, valor#2L], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_valor#23L ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(total_valor#23L ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=69]
      +- HashAggregate(keys=[regiao

In [10]:
dados = [
    ("JP", "Arthur", 100),
    ("CG", "Bruna", 200),
    ("JP", "Carlos", 50),
    ("JP", "Daniel", 70),
    ("CG", "Camila", 150),
    ("PE", "Eduardo", 120),
    ("PE", "Fernanda", 130),
]

df2 = spark.createDataFrame(dados, ["regiao", "nome", "valor"])

In [11]:
df2.show()

26/09/16 20:25:59 INFO CodeGenerator: Code generated in 10.638937 ms
26/09/16 20:25:59 INFO SparkContext: Starting job: showString at <unknown>:0
26/09/16 20:25:59 INFO DAGScheduler: Got job 2 (showString at <unknown>:0) with 1 output partitions
26/09/16 20:25:59 INFO DAGScheduler: Final stage: ResultStage 3 (showString at <unknown>:0)
26/09/16 20:25:59 INFO DAGScheduler: Parents of final stage: List()
26/09/16 20:25:59 INFO DAGScheduler: Missing parents: List()
26/09/16 20:25:59 INFO DAGScheduler: Submitting ResultStage 3 (MapPartitionsRDD[17] at showString at <unknown>:0), which has no missing parents
26/09/16 20:25:59 INFO MemoryStore: Block broadcast_2 stored as values in memory (estimated size 13.0 KiB, free 1048.8 MiB)
26/09/16 20:25:59 INFO MemoryStore: Block broadcast_2_piece0 stored as bytes in memory (estimated size 6.7 KiB, free 1048.8 MiB)
26/09/16 20:25:59 INFO BlockManagerInfo: Added broadcast_2_piece0 in memory on jupyter:37861 (size: 6.7 KiB, free: 1048.8 MiB)
26/09/16 

+------+--------+-----+
|regiao|    nome|valor|
+------+--------+-----+
|    JP|  Arthur|  100|
|    CG|   Bruna|  200|
|    JP|  Carlos|   50|
|    JP|  Daniel|   70|
|    CG|  Camila|  150|
|    PE| Eduardo|  120|
|    PE|Fernanda|  130|
+------+--------+-----+



26/09/16 20:25:59 INFO TaskSetManager: Finished task 2.0 in stage 4.0 (TID 23) in 65 ms on jupyter (executor driver) (2/4)
26/09/16 20:25:59 INFO TaskSetManager: Finished task 3.0 in stage 4.0 (TID 24) in 65 ms on jupyter (executor driver) (3/4)
26/09/16 20:25:59 INFO PythonRunner: Times: total = 62, boot = 16, init = 46, finish = 0
26/09/16 20:25:59 INFO Executor: Finished task 1.0 in stage 4.0 (TID 22). 1942 bytes result sent to driver
26/09/16 20:25:59 INFO TaskSetManager: Finished task 1.0 in stage 4.0 (TID 22) in 77 ms on jupyter (executor driver) (4/4)
26/09/16 20:25:59 INFO TaskSchedulerImpl: Removed TaskSet 4.0, whose tasks have all completed, from pool 
26/09/16 20:25:59 INFO DAGScheduler: ResultStage 4 (showString at <unknown>:0) finished in 0.088 s
26/09/16 20:25:59 INFO DAGScheduler: Job 3 is finished. Cancelling potential speculative or zombie tasks for this job
26/09/16 20:25:59 INFO TaskSchedulerImpl: Killing all running tasks in stage 4: Stage finished
26/09/16 20:25:59

In [15]:
df_uf = df2.filter(F.col("regiao") == "JP")

In [16]:
df_duplicado = df_uf.filter(F.col("nome") == "Arthur")

In [17]:
df_uf.explain(True) 

== Parsed Logical Plan ==
'Filter ('regiao = JP)
+- LogicalRDD [regiao#28, nome#29, valor#30L], false

== Analyzed Logical Plan ==
regiao: string, nome: string, valor: bigint
Filter (regiao#28 = JP)
+- LogicalRDD [regiao#28, nome#29, valor#30L], false

== Optimized Logical Plan ==
Filter (isnotnull(regiao#28) AND (regiao#28 = JP))
+- LogicalRDD [regiao#28, nome#29, valor#30L], false

== Physical Plan ==
*(1) Filter (isnotnull(regiao#28) AND (regiao#28 = JP))
+- *(1) Scan ExistingRDD[regiao#28,nome#29,valor#30L]



In [22]:
df_filtrado2 = (df2
                .filter((F.col("regiao") == "JP") & (F.col("nome") == "Arthur"))
               )

In [23]:
df_filtrado2.explain(True)

== Parsed Logical Plan ==
'Filter (('regiao = JP) AND ('nome = Arthur))
+- LogicalRDD [regiao#28, nome#29, valor#30L], false

== Analyzed Logical Plan ==
regiao: string, nome: string, valor: bigint
Filter ((regiao#28 = JP) AND (nome#29 = Arthur))
+- LogicalRDD [regiao#28, nome#29, valor#30L], false

== Optimized Logical Plan ==
Filter ((isnotnull(regiao#28) AND isnotnull(nome#29)) AND ((regiao#28 = JP) AND (nome#29 = Arthur)))
+- LogicalRDD [regiao#28, nome#29, valor#30L], false

== Physical Plan ==
*(1) Filter ((isnotnull(regiao#28) AND isnotnull(nome#29)) AND ((regiao#28 = JP) AND (nome#29 = Arthur)))
+- *(1) Scan ExistingRDD[regiao#28,nome#29,valor#30L]



In [24]:
df_filtrado3 = (df2
                .filter(F.col("regiao") == "JP") 
                .filter(F.col("nome") == "Arthur")
               )

In [ ]:
df_filtrado3.explain(True)

== Parsed Logical Plan ==
'Filter ('nome = Arthur)
+- Filter (regiao#28 = JP)
   +- LogicalRDD [regiao#28, nome#29, valor#30L], false

== Analyzed Logical Plan ==
regiao: string, nome: string, valor: bigint
Filter (nome#29 = Arthur)
+- Filter (regiao#28 = JP)
   +- LogicalRDD [regiao#28, nome#29, valor#30L], false

== Optimized Logical Plan ==
Filter ((isnotnull(regiao#28) AND isnotnull(nome#29)) AND ((regiao#28 = JP) AND (nome#29 = Arthur)))
+- LogicalRDD [regiao#28, nome#29, valor#30L], false

== Physical Plan ==
*(1) Filter ((isnotnull(regiao#28) AND isnotnull(nome#29)) AND ((regiao#28 = JP) AND (nome#29 = Arthur)))
+- *(1) Scan ExistingRDD[regiao#28,nome#29,valor#30L]



26/09/17 00:37:54 INFO SparkContext: Invoking stop() from shutdown hook
26/09/17 00:37:54 INFO SparkContext: SparkContext is stopping with exitCode 0.
26/09/17 00:37:54 INFO AbstractConnector: Stopped Spark@592eb5a4{HTTP/1.1, (http/1.1)}{0.0.0.0:4040}
26/09/17 00:37:54 INFO SparkUI: Stopped Spark web UI at http://jupyter:4040
26/09/17 00:37:54 INFO MapOutputTrackerMasterEndpoint: MapOutputTrackerMasterEndpoint stopped!
26/09/17 00:37:54 INFO MemoryStore: MemoryStore cleared
26/09/17 00:37:54 INFO BlockManager: BlockManager stopped
26/09/17 00:37:54 INFO BlockManagerMaster: BlockManagerMaster stopped
26/09/17 00:37:54 INFO OutputCommitCoordinator$OutputCommitCoordinatorEndpoint: OutputCommitCoordinator stopped!
26/09/17 00:37:54 INFO SparkContext: Successfully stopped SparkContext
26/09/17 00:37:54 INFO ShutdownHookManager: Shutdown hook called
26/09/17 00:37:54 INFO ShutdownHookManager: Deleting directory /tmp/spark-7e309d11-f452-45b5-b08d-1177436091e5/pyspark-920845fd-9fe3-4727-958c-4